**Calculate Correlation Coefficient between station observations and incident pixels**


In [37]:
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import hydroeval as hel

In [38]:
#Read insitu data
df_insitu = pd.read_csv(r'..\Data\Processed\Insitu\insitu_rainfall_1990_2019.csv')
df_insitu['date'] = pd.to_datetime(df_insitu[['year', 'month', 'day']])

#Read gridded data - starting with era5L
era5L_path = r"..\Data\Datasets\ERA5_land_prec"
era5L_tiff_files = glob.glob(era5L_path + "/*.tif")
#era5L_tiff_files

#helper function - extract date from filename
def extract_date(file):
    filename = os.path.basename(file)
    return filename[11:19]

era5L_dates = list(map(extract_date, era5L_tiff_files))
#era5L_dates

In [39]:
#extract station lat and lon in a list 
unique_stn = df_insitu[['index_no','lat','long']].drop_duplicates()
stn_points = list(zip(unique_stn['long'],unique_stn['lat']))
#stn_points
unique_stn   

,index_no,lat,long
0,102,29.55744,80.41317
10804,103,29.46714,80.54575
21761,104,29.30135,80.58775
32718,105,28.95477,80.23049
43207,107,29.84496,80.54539
...,...,...,...
852821,1407,26.90964,87.92308
862846,1408,26.67067,87.70318
873709,1412,26.56389,88.06150
884545,1416,26.86806,88.07833


In [40]:
#extract values from tiff files using rasterio
def process_file(file):
    date_obj = extract_date(file)
    local_records = []
    try:
        with rasterio.open(file) as src:
            crs = src.crs
            if crs == "EPSG:4326":
                sampled_values = list(src.sample(stn_points))
                
                for i, value in enumerate(sampled_values):
                    local_records.append({
                        'date': date_obj,
                        'stn_id': unique_stn.iloc[i]['index_no'],
                        'prec_grid': value[0]
                        })
                 
    except Exception as e:
        print(f'Error processing {file}: {e}')
    return local_records

prec_records = []
with concurrent.futures.ThreadPoolExecutor() as executor:
    results = executor.map(process_file, era5L_tiff_files)
    for result in results:
        prec_records.extend(result)

df_grid = pd.DataFrame(prec_records)
df_grid

,date,stn_id,prec_grid
0,19900101,102.0,0.000438
1,19900101,103.0,0.000423
2,19900101,104.0,0.000423
3,19900101,105.0,0.000438
4,19900101,107.0,0.000423
...,...,...,...
942211,20191231,1407.0,0.078398
942212,20191231,1408.0,0.000852
942213,20191231,1412.0,0.000852
942214,20191231,1416.0,0.015113


In [41]:
#merging station and grid df into a single df
df_insitu1 = df_insitu[['date','index_no','rainfall_sum']]
df_insitu1 = df_insitu1.rename(columns={'rainfall_sum':'prec_insitu', 'index_no':'stn_id'})

df_grid['date'] = pd.to_datetime(df_grid['date'])
df_insitu1['date'] = pd.to_datetime(df_insitu1['date'])
df_merged = pd.merge(df_grid, df_insitu1, on=['date','stn_id'], how='inner')
df_merged = df_merged.dropna()
df_merged = df_merged.round(1)
print(df_merged.shape)

(710697, 4)


**Calculating error metrices - Example ERA5L**

In [ ]:
#Convert to array
prec_grid_array = np.array(df_merged['prec_grid'])
prec_insitu_array = np.array(df_merged['prec_insitu'])

#check data range
print("Insitu data range:", np.min(prec_insitu_array), "to", np.max(prec_insitu_array))
print("Grid data range:", np.min(prec_grid_array), "to", np.max(prec_grid_array))

#convert negative value in gridded data to zero
prec_grid_array[prec_grid_array < 0] = 0

#correlation coefficient
correlation_era5L = he.spearman_r(prec_grid_array, prec_insitu_array)
print("Correlation (Spearman) Value: ", correlation_era5L)

#NRMSE
nrmse_era5L = he.nrmse_mean(prec_grid_array, prec_insitu_array)
print("NRMSE: ", nrmse_era5L)

#Percent bias
pbias_era5L = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
print("Percent Bias: ", pbias_era5L)

#Relative standard deviation 
rsd_era5L = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
print("Relative Standard Deviation: ", rsd_era5L)

Insitu data range: 0.0 to 403.2
Grid data range: 0.0 to 478.3
Correlation (Spearman) Value:  0.5790873054725064
NRMSE:  3.2480134275681163
Percent Bias:  35.633047964772736
Relative Standard Deviation:  1.033309665178169


**Code for CPI Calculation after accuracy metrices are calculated for individual datasets**

In [54]:
# Example values of accuracy metrics for other datasets - MSWEP, HAR, CHIRPS, PERSIANN
correlation_mswep, nrmse_mswep, pbias_mswep, rsd_mswep = 0.6846, 2.1397, -8.1814, 1.0509
correlation_har, nrmse_har, pbias_har, rsd_har = 0.235, 0.8973, 12.4567, 0.1234
correlation_chirps, nrmse_chirps, pbias_chirps, rsd_chirps = 0.49, 5.4521, -5.2341, 1.8876
correlation_persiann, nrmse_persiann, pbias_persiann, rsd_persiann = 0.1734, 3.1245, -18.3456, 1.1767

#create a dataframe of the accuracy metrics
df_accuracy = pd.DataFrame({
    'Dataset': ['MSWEP', 'HAR', 'CHIRPS', 'PERSIANN', 'ERA5L'],
    'Correlation': [correlation_mswep, correlation_har, correlation_chirps, correlation_persiann, correlation_era5L],
    'NRMSE': [nrmse_mswep, nrmse_har, nrmse_chirps, nrmse_persiann, nrmse_era5L],
    'PBIAS': [pbias_mswep, pbias_har, pbias_chirps, pbias_persiann, pbias_era5L],
    'RSD': [rsd_mswep, rsd_har, rsd_chirps, rsd_persiann, rsd_era5L]
})


#Normalization
df_accuracy['Abs_PBIAS'] = df_accuracy['PBIAS'].abs()

def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())

metrics = ['Correlation', 'NRMSE', 'Abs_PBIAS', 'RSD']

df_accuracy_normalized = df_accuracy.copy().drop(columns=['PBIAS'])
for metric in metrics:
    df_accuracy_normalized[metric] = normalize_column(df_accuracy_normalized[metric])

#converting to scale of 0 to 1 (ideal to worst) - only correlation needs inversion
df_accuracy_normalized['Correlation'] = 1 - df_accuracy_normalized['Correlation']
df_accuracy_normalized
 
#get cpi for each (linear)
df_accuracy_normalized['CPI'] = df_accuracy_normalized['Correlation'] + df_accuracy_normalized['NRMSE'] + df_accuracy_normalized['Abs_PBIAS'] + df_accuracy_normalized['RSD']
df_accuracy_normalized['CPI_RANK'] = df_accuracy_normalized['CPI'].rank(ascending=True)

#retain final df with cpi_rank and actual values of correlation, nrmse, pbias, rsd
df_accuracy_final = pd.DataFrame({
    'Dataset': df_accuracy['Dataset'],
    'Correlation': df_accuracy['Correlation'],
    'NRMSE': df_accuracy['NRMSE'],
    'PBIAS': df_accuracy['PBIAS'],
    'RSD': df_accuracy['RSD'],
    'CPI_RANK': df_accuracy_normalized['CPI_RANK']
})
df_accuracy_final

,Dataset,Correlation,NRMSE,PBIAS,RSD,CPI_RANK
0,MSWEP,0.684600,2.139700,-8.181400,1.05090,1.0
1,HAR,0.235000,0.897300,12.456700,0.12340,2.0
2,CHIRPS,0.490000,5.452100,-5.234100,1.88760,4.0
3,PERSIANN,0.173400,3.124500,-18.345600,1.17670,5.0
4,ERA5L,0.579087,3.248013,35.633048,1.03331,3.0
